<p style ="text-align:center">
    <img src="http://epecora.com.br/DataFiles/BannerUFPR.png" width="700" alt="PPGOLD/PPGMNE Python:INTRO"  />
</p>

# Data Science for Business

## Prof. Eduardo Pécora

## Scalling
Tempo estimado: **30** minutos

## Objetivos

Após completar esta aula, você será capaz de:

* Entender o processo de Scalling
* Usar piplines

## Bibliotecas

In [1]:
# importando a biblioteca pandas para manipulação de dados
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

# importando as bibliotecas do matplotlib para gerar gráficos
import matplotlib as mpl
import matplotlib.pyplot as plt

# Importando a biblioteca numpy e math que fornece funções matemáticas básicas
import numpy as np
import math

# Importando biblioteca do seaborn para gerar gráficos mais atraentes e informativos
import seaborn as sns

# Usado para exibir os gráficos gerados pela biblioteca Matplotlib diretamente no notebook, sem precisar abrir uma janela externa.
%matplotlib inline


## Coletando dados

In [2]:
# Caminho do arquivo csv
caminho = "https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Dados/MeuAutoLimpo.csv"

#Obtendo arquivo e passando-o para um dataframe
df = pd.read_csv(caminho)

## Preparação de dados

In [3]:
df_dummy = df.copy()

label_binarizer = LabelBinarizer()
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html

dicotomica = df_dummy['num-of-doors']
label_binarizer.fit(dicotomica)
df_dummy['num-of-doors'] = label_binarizer.transform(dicotomica)

# renomeando as marcas
df_dummy['make'] = df_dummy['make'].replace({'bmw':'BMW', 'doge': 'dodge', 'VW':'volkswagen', 'volv1':'volvo' })

# Crie uma instância do codificador OneHotEncoder
encoder = OneHotEncoder()

# Ajuste e transforme os dados da coluna "make"
poly_array = encoder.fit_transform(df_dummy[['make']])

# Crie um DataFrame Pandas com as variáveis dummy
poly_df = pd.DataFrame(poly_array.toarray(), columns=encoder.get_feature_names_out(['make']))

# Concatene o DataFrame dummy com os outros dados
df_dummy = pd.concat([df_dummy.drop('make', axis=1), poly_df], axis=1)

df_dummy.drop(['fuel-type', 'aspiration', 'body-style', 'drive-wheels', 
               'engine-location', 'engine-type', 'num-of-cylinders', 'fuel-system'], axis =1, inplace = True)
df_dummy.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 39 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   symboling           201 non-null    int64  
 1   normalized-losses   201 non-null    int64  
 2   num-of-doors        201 non-null    int64  
 3   wheel-base          201 non-null    float64
 4   length              201 non-null    float64
 5   width               201 non-null    float64
 6   height              201 non-null    float64
 7   curb-weight         201 non-null    int64  
 8   engine-size         201 non-null    int64  
 9   bore                201 non-null    float64
 10  stroke              201 non-null    float64
 11  compression-ratio   201 non-null    float64
 12  horsepower          201 non-null    float64
 13  peak-rpm            201 non-null    float64
 14  city-mpg            201 non-null    int64  
 15  highway-mpg         201 non-null    int64  
 16  price              

# Variáveis Dependentes e Independentes

In [4]:
# Separate features and target variable
X = df_dummy.drop(columns=['price'])
y = df_dummy['price']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scalling

## Importância do Scaling em Modelos de Regressão

Modelos como Ridge, Lasso e Elastic Net utilizam penalização nos coeficientes. Essas penalizações dependem da escala das variáveis.

Se uma variável estiver em uma escala muito maior que outra, ela pode dominar o modelo, prejudicando a interpretação e o desempenho.

Neste notebook, vamos comparar o impacto de diferentes técnicas de scaling nos resultados dos modelos.

## Função de avaliação (evita repetição e deixa o código mais limpo)

In [5]:
def avaliar_modelo(modelo, X_train, X_test, y_train, y_test):
    modelo.fit(X_train, y_train)

    y_pred_train = modelo.predict(X_train)
    y_pred_test = modelo.predict(X_test)

    return {
        "R2 Treino": r2_score(y_train, y_pred_train),
        "R2 Teste": r2_score(y_test, y_pred_test),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test))
    }

## Modelos

In [6]:
modelos = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
}

resultados_sem_scaling = []

## Sem Scaling

In [7]:
for nome, modelo in modelos.items():
    res = avaliar_modelo(modelo, X_train, X_test, y_train, y_test)
    res["Modelo"] = nome
    res["Scaling"] = "Sem Scaling"
    resultados_sem_scaling.append(res)

df_sem_scaling = pd.DataFrame(resultados_sem_scaling)
df_sem_scaling = df_sem_scaling.round(4)
df_sem_scaling

,R2 Treino,R2 Teste,RMSE,Modelo,Scaling
0,0.9311,0.9024,3245.7691,Ridge,Sem Scaling
1,0.9449,0.9106,3105.8892,Lasso,Sem Scaling
2,0.8476,0.7928,4728.8038,ElasticNet,Sem Scaling


## Com Scaling

In [8]:
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
    "MaxAbsScaler": MaxAbsScaler()
}

def testar_scalers(scalers, modelos):
    resultados = []

    for nome_scaler, scaler in scalers.items():
        # fit só no treino
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        for nome_modelo, modelo in modelos.items():
            res = avaliar_modelo(
                modelo,
                X_train_scaled,
                X_test_scaled,
                y_train,
                y_test
            )

            res["Modelo"] = nome_modelo
            res["Scaling"] = nome_scaler
            resultados.append(res)

    return pd.DataFrame(resultados)

In [9]:
df_com_scaling = testar_scalers(scalers, modelos)
df_com_scaling.round(4)
df_com_scaling

,R2 Treino,R2 Teste,RMSE,Modelo,Scaling
0,0.944771,0.909108,3131.840325,Ridge,StandardScaler
1,0.944927,0.909450,3125.938765,Lasso,StandardScaler
2,0.917591,0.878887,3615.210150,ElasticNet,StandardScaler
3,0.927956,0.889358,3455.387834,Ridge,MinMaxScaler
4,0.944842,0.910551,3106.883536,Lasso,MinMaxScaler
5,0.407819,0.288705,8761.172734,ElasticNet,MinMaxScaler
6,0.931041,0.901211,3265.067212,Ridge,RobustScaler
7,0.944889,0.910591,3106.194700,Lasso,RobustScaler
8,0.815732,0.715925,5536.737370,ElasticNet,RobustScaler
9,0.920067,0.885592,3513.701930,Ridge,MaxAbsScaler


In [10]:
df_final = pd.concat([df_sem_scaling, df_com_scaling], ignore_index=True)

df_final.sort_values(by=["Modelo","R2 Teste"], ascending=[True,False]).reset_index(drop=True)

,R2 Treino,R2 Teste,RMSE,Modelo,Scaling
0,0.917591,0.878887,3615.210150,ElasticNet,StandardScaler
1,0.847600,0.792800,4728.803800,ElasticNet,Sem Scaling
2,0.815732,0.715925,5536.737370,ElasticNet,RobustScaler
3,0.407819,0.288705,8761.172734,ElasticNet,MinMaxScaler
4,0.245875,0.152927,9560.880735,ElasticNet,MaxAbsScaler
5,0.944218,0.911876,3083.791919,Lasso,MaxAbsScaler
6,0.944900,0.910600,3105.889200,Lasso,Sem Scaling
7,0.944889,0.910591,3106.194700,Lasso,RobustScaler
8,0.944842,0.910551,3106.883536,Lasso,MinMaxScaler
9,0.944927,0.909450,3125.938765,Lasso,StandardScaler


# Previsão com Scaler

In [11]:
# Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelo
model = Ridge()
model.fit(X_train_scaled, y_train)

# Dado Novo
novo = X.iloc[[14]]   # copia uma linha qq do DF para simular um dado novo.
novo_scaled = scaler.transform(novo) # Previsão com o dado APÓS a aplicação do scaler 

# Previsão
previsao = model.predict(novo_scaled)
print(previsao)

[27119.66083838]


# Ganhando em escala com PIPELINE

In [12]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

pipe.fit(X_train, y_train)
pipe.predict(novo)

array([27119.66083838])

## Fique Conectado

- [![YouTube](https://img.icons8.com/?size=40&id=19318&format=png&color=000000)](https://www.youtube.com/@LigaDataScience/videos)  
  Explore nossos vídeos educacionais e webinars sobre ciência de dados, machine learning e inteligência artificial. Inscreva-se para não perder nenhuma atualização!

- [![LinkedIn](https://img.icons8.com/?size=40&id=13930&format=png&color=000000)](https://www.linkedin.com/company/liga-data-science-ufpr/)  
  Siga-nos no LinkedIn para as últimas novidades, oportunidades de carreira e networking profissional no campo da ciência de dados.

- [![Instagram](https://img.icons8.com/?size=40&id=32323&format=png&color=000000)](https://www.instagram.com/ligadatascience/)  
  Confira nosso Instagram para conteúdos dos bastidores, destaques de eventos e o dia a dia da Liga Data Science. Faça parte da nossa jornada!

## Referências:

* Documentação da biblioteca <a href="https://pandas.pydata.org/docs/">Pandas</a>
* Documentação do método <a href=https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html>train_test_split</a>


## Autores

<a href="https://www.linkedin.com/in/eduardopecora/" target="_blank">Eduardo Pecora</a>


## Log de modificações

| Data | Versão | Modificado por | Descrição |
| -----------| ------- | ---------- | ---------------------------------- |
| 06-04-2026       | 1.0   | Eduardo Pecora    | Estrutura Aula        |


## <h3 align="center"> (c) Liga Data Science / UFPR 2026. All rights reserved. <h3/>